In [1]:
import os.path as op 
import os
from glob import glob
import pickle
import pingouin as pg
import seaborn as sns
import pandas as pd
import numpy as np

In [2]:
qa_df = pd.read_csv('/global/homes/m/mphagen/functional-connectivity/connectome-comparison/results/2026_05_26_absolute_movement.csv', 
                    index_col=0).sort_index()
qa_df.index = [f'sub-{i}' for i in qa_df.index] 
print(qa_df.shape, qa_df.columns)

(1113, 4) Index(['RL_1', 'RL_2', 'LR_2', 'LR_1'], dtype='str')


In [3]:
old_qa_df = pd.read_csv('/global/homes/m/mphagen/functional-connectivity/connectome-comparison/data/absolute_movement.csv')

In [4]:
old_qa_df.index = [f'sub-{i}' for i in old_qa_df.index] 


In [5]:
old_qa_df = old_qa_df.drop('Unnamed: 0', axis=1) 

In [6]:
#adopt future option 
pd.set_option('future.no_silent_downcasting', True)

unrestricted_df = pd.read_csv('../../../data/unrestricted_mphagen_1_27_2022_20_50_7.csv')
unrestricted_df.index = [f'sub-{i}' for i in unrestricted_df['Subject']]
unrestricted_df[['Gender']] = unrestricted_df[['Gender']].replace({'M':0, 'F':1}).infer_objects(copy=False)

/tmp/ipykernel_1960569/831740102.py:2: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)
/tmp/ipykernel_1960569/831740102.py:6: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  unrestricted_df[['Gender']] = unrestricted_df[['Gender']].replace({'M':0, 'F':1}).infer_objects(copy=False)


In [7]:
restricted_df = pd.read_csv('../../../data/RESTRICTED_arokem_1_31_2022_23_26_45.csv')
restricted_df.index = [f'sub-{i}' for i in restricted_df['Subject']]

In [8]:
covariates = ['Gender', 'Age_in_Yrs']
covariate_df = unrestricted_df[['Gender']].join(restricted_df[['Age_in_Yrs']]) 

In [9]:
def make_fc_df(results_dict, ses): 
    """flatten connectomes from dict keys """
    fc_df = pd.DataFrame(columns=results_dict.keys()) 
    
    for key in results_dict.keys(): 
        fc_df[key] = results_dict[key][f'ses-{ses}'].flatten() 
    fc_df = fc_df.T
    return fc_df

In [10]:
def make_qcfc_df(fc_df, cov_df, qa_df, ses): 
    """join df """
    qa_df[[f'RL_{ses}', f'LR_{ses}']]
    connectomes = fc_df.join(covariate_df) 
    rms_df = qa_df[[f'RL_{ses}', f'LR_{ses}']].mean(axis=1).rename('rms')
    connectomes = connectomes.join(rms_df)
    return connectomes

In [11]:
def qcfc(qcfc_df, covariates, edge_range): 
    result_df = pd.DataFrame(index=edge_range, columns=['r',  'p-val'])
    result_df.index = [f'edge-{i}' for i in result_df.index]

    results_list = {}
    for edge_id in edge_range:
    # QC-FC
        metric = pg.partial_corr(data=qcfc_df, 
                    x=edge_id, 
                    y='rms', 
                    covar=covariates)

        result_df.loc[f'edge-{edge_id}'] = metric.loc['pearson', ['r', 'p-val']]
    return result_df

In [12]:
def run_qcfc(model, proc_type, ses, boot=True, date_str='2026-05-22'): 
    result_dict = {} 
    with open(glob(op.join(results_path, proc_type, f'*{date_str}*{model}*relmat.pkl'))[0], 'rb') as l:
        connectome_dict = pickle.load(l)
    for ii in connectome_dict.keys(): 
        del connectome_dict[ii]['ses-full']
    fc_df = make_fc_df(connectome_dict, ses=ses)
    edge_range = fc_df.columns

    qcfc_df = make_qcfc_df(fc_df, covariate_df, qa_df, ses=ses)
    ## add resampling here        
        
    result_dict[f'{model}_{proc_type}_ses-{ses}'] = qcfc(qcfc_df, covariates, edge_range)
    return result_dict

In [13]:
qcfc_dict = {} 

In [14]:
results_path = '/global/homes/m/mphagen/functional-connectivity/connectome-comparison/results'

In [15]:
import warnings
# Or ignore specific categories
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [16]:
date_str = '2026-05-22'

In [17]:
# model = 'lassoBIC'
# proc_type = 'MSMAll'
# qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='2')) 

In [18]:
model = 'lassoBIC'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026-05-22', ses='1')) 
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026-05-22', ses='2')) 

In [19]:
model = '-correlation'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026-06-01', ses='1')) 
qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026-06-01', ses='2')) 


In [20]:
# model = 'lassoBIC'
# proc_type = 'xcpdPNC'
# qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, date_str='2026', ses='2'))

In [21]:
# model = '-correlation'
# proc_type = 'MSMAll'
# qcfc_dict.update(run_qcfc(model, proc_type, date_str= '2026-05-01', ses='1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, date_str= '2026-05-01',ses='2')) 

In [22]:
# model = '-correlation'
# proc_type = 'xcpd'
# qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [23]:
model = 'lassoBIC'
proc_type = 'xcpd'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [24]:
model = 'ridgeCV'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [25]:
model = 'partial'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [26]:
model = 'uoi'
proc_type = 'MSMAll_FIX'
qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [27]:
# model = 'corr-lassoThresh'
# proc_type = 'xcpd'
# qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [28]:
# model = 'corr-lassoThresh'
# proc_type = 'MSMAll'
# qcfc_dict.update(run_qcfc(model, proc_type, '1')) 
# qcfc_dict.update(run_qcfc(model, proc_type, '2')) 

In [29]:
import datetime

In [30]:
with open(f'../02-Analysis/results/{str(datetime.date.today())}_qcfc_dict.pkl', 'wb') as file: 
    pickle.dump(qcfc_dict, file) 

In [31]:
mask = np.triu(np.ones_like(np.ones((100,100)), dtype=bool), k=1) 

In [32]:
df = qcfc_dict['-correlation_MSMAll_ses-1']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

KeyError: '-correlation_MSMAll_ses-1'

In [ ]:
df = qcfc_dict['-correlation_MSMAll_ses-2']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['ridgeCV_MSMAll_FIX_ses-1']
# df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['ridgeCV_MSMAll_FIX_ses-2']
# df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['-correlation_MSMAll_FIX_ses-2']
df = df.loc[mask.ravel()]

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['lassoBIC_MSMAll_FIX_ses-1']

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['lassoBIC_MSMAll_FIX_ses-2']

df.loc[df['p-val'] <= .05].shape

In [ ]:
df = qcfc_dict['-correlation_xcpd_ses-1']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['corr-lassoThresh_xcpd_ses-2']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['-correlation_xcpd_ses-2']
df = df.loc[mask.ravel()]
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['lassoBIC_xcpd_ses-1']
df.loc[df['p-val'] <= .05] 

In [ ]:
df = qcfc_dict['lassoBIC_xcpd_ses-2']
df.loc[df['p-val'] <= .05] 